# Comprehensive Federated Learning Pipeline (Split Execution)
This notebook evaluates multiple models (MobileNet, EfficientNet, YOLO) and aggregation strategies (FedAvg, FedWeightedAvg, FedProx) on IID and Non-IID data.

**Key Improvements:**
- **Transfer Learning**: Enabled pre-trained ImageNet weights (`weights='DEFAULT'` / `.pt` files).
- **Data Augmentation**: Added Random Horizontal/Vertical Flips and Rotations.
- **Split Execution Blocks**: The main training loop is split into 3 independent cells (MobileNet, EfficientNet, YOLO) to prevent total data loss if Colab disconnects.



In [ ]:
!pip install ultralytics kagglehub scikit-learn seaborn -q


In [ ]:
import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using Device:', DEVICE)

# Download Dataset
path = kagglehub.dataset_download("prahladmehandiratta/cervical-cancer-largest-dataset-sipakmed")
DATASET_ROOT = os.path.join(path)
print("Dataset path:", DATASET_ROOT)

# Transforms & Load Dataset (WITH DATA AUGMENTATION)
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

full_dataset = datasets.ImageFolder(root=DATASET_ROOT, transform=transform_train)
NUM_CLASSES = len(full_dataset.classes)
print(f"Found {NUM_CLASSES} classes: {full_dataset.classes}")



In [ ]:
def create_iid_splits(dataset, num_clients):
    indices = np.random.permutation(len(dataset))
    split_size = len(dataset) // num_clients
    return [Subset(dataset, indices[i * split_size : (i + 1) * split_size]) for i in range(num_clients)]

def create_noniid_splits(dataset, num_clients, alpha=0.5):
    targets = np.array([full_dataset.targets[dataset.indices[i]] for i in range(len(dataset.indices))])
    client_indices = [[] for _ in range(num_clients)]
    classes = np.unique(targets)

    for c in classes:
        idx = np.where(targets == c)[0]
        np.random.shuffle(idx)
        proportions = np.random.dirichlet(np.repeat(alpha, num_clients))
        proportions = np.array([p * len(idx) for p in proportions]).astype(int)
        proportions[-1] += len(idx) - proportions.sum()

        ptr = 0
        for i, p in enumerate(proportions):
            client_indices[i].extend(idx[ptr:ptr+p].tolist())
            ptr += p

    actual_indices = np.array(dataset.indices)
    return [Subset(dataset.dataset, actual_indices[idx]) for idx in client_indices]

train_idx, test_idx = train_test_split(np.arange(len(full_dataset)), test_size=0.2, random_state=42, stratify=full_dataset.targets)
train_dataset = Subset(full_dataset, train_idx)

test_dataset = datasets.ImageFolder(root=DATASET_ROOT, transform=transform_test)
test_dataset = Subset(test_dataset, test_idx)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

NUM_CLIENTS = 3
clients_iid = create_iid_splits(train_dataset, NUM_CLIENTS)
clients_noniid = create_noniid_splits(train_dataset, NUM_CLIENTS, alpha=0.5)

print(f"Train samples: {len(train_dataset)} | Test samples: {len(test_dataset)}")



In [ ]:
def get_model(model_name):
    if model_name == 'mobilenet':
        model = models.mobilenet_v3_small(weights='DEFAULT')
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, NUM_CLASSES)
    elif model_name == 'efficientnet':
        model = models.efficientnet_b0(weights='DEFAULT')
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
    elif model_name == 'yolo':
        from ultralytics import YOLO
        yolo = YOLO('yolov8n-cls.pt', verbose=False)
        model = yolo.model
        model.model[-1].linear = nn.Linear(model.model[-1].linear.in_features, NUM_CLASSES)
    else:
        raise ValueError("Unsupported model architecture")
    return model.to(DEVICE)



In [ ]:
def train_client(model, dataloader, epochs=1, lr=1e-4, fedprox_mu=0.0, global_model=None):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    if global_model is not None:
        global_model.eval()

    for epoch in range(epochs):
        for data, target in dataloader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            optimizer.zero_grad()
            output = model(data)

            if isinstance(output, tuple):
                output = output[0]

            loss = criterion(output, target)

            if fedprox_mu > 0 and global_model is not None:
                prox_term = 0.0
                for w, w_t in zip(model.parameters(), global_model.parameters()):
                    prox_term += (w - w_t).norm(2)
                loss += (fedprox_mu / 2) * prox_term

            loss.backward()
            optimizer.step()

    return {k: v.cpu().clone() for k, v in model.state_dict().items()}

def aggregate_models(local_weights, strategy, client_sizes):
    global_weights = copy.deepcopy(local_weights[0])
    total_samples = sum(client_sizes)

    if strategy == 'fedavg_unweighted':
        weights = [1.0 / len(local_weights)] * len(local_weights)
    elif strategy in ['fedavg_weighted', 'fedprox']:
        weights = [size / total_samples for size in client_sizes]

    for key in global_weights.keys():
        temp = torch.zeros_like(global_weights[key], dtype=torch.float32)
        for i in range(len(local_weights)):
            temp += local_weights[i][key].float() * weights[i]
        global_weights[key] = temp.to(global_weights[key].dtype)

    return global_weights

def evaluate(model, dataloader):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for data, target in dataloader:
            output = model(data.to(DEVICE))
            if isinstance(output, tuple):
                output = output[0]
            preds = output.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(target.numpy())

    acc = np.mean(np.array(all_preds) == np.array(all_targets))
    macro_prec = precision_score(all_targets, all_preds, average='macro', zero_division=0)
    macro_rec = recall_score(all_targets, all_preds, average='macro', zero_division=0)
    macro_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)

    cm = confusion_matrix(all_targets, all_preds)
    specificities = []
    for i in range(len(cm)):
        tn = np.sum(cm) - (np.sum(cm[i, :]) + np.sum(cm[:, i]) - cm[i, i])
        fp = np.sum(cm[:, i]) - cm[i, i]
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        specificities.append(spec)

    macro_spec = np.mean(specificities)
    return acc, macro_prec, macro_rec, macro_spec, macro_f1



## Part 1: Main Experiment Settings


In [ ]:
# Initialize the global configurations and result tracking list
results = []
NUM_ROUNDS = 10
LOCAL_EPOCHS = 2

strategies = ['fedavg_unweighted', 'fedavg_weighted', 'fedprox']
distributions = {'iid': clients_iid, 'non_iid': clients_noniid}

print("Configuration initialized! Ready to run individual models.")



## Part 1A: MobileNet Experiments


In [ ]:
# ==========================================
# EXPERIMENT 1: MOBILENET
# ==========================================
model_name = 'mobilenet'

for dist_name, client_datasets in distributions.items():
    client_loaders = [DataLoader(ds, batch_size=32, shuffle=True) for ds in client_datasets]
    client_sizes = [len(ds) for ds in client_datasets]

    for strategy in strategies:
        print(f"\n--- Running | Model: {model_name.upper()} | Dist: {dist_name.upper()} | Strategy: {strategy.upper()} ---")

        global_model = get_model(model_name)
        global_weights = {k: v.cpu() for k, v in global_model.state_dict().items()}
        fedprox_mu = 0.01 if strategy == 'fedprox' else 0.0

        for rnd in range(NUM_ROUNDS):
            local_weights = []
            for loader in client_loaders:
                local_model = get_model(model_name)
                local_model.load_state_dict(global_weights)
                weights = train_client(local_model, loader, epochs=LOCAL_EPOCHS, lr=2e-4, fedprox_mu=fedprox_mu, global_model=global_model)
                local_weights.append(weights)

            global_weights = aggregate_models(local_weights, strategy, client_sizes)
            global_model.load_state_dict(global_weights)

        global_model.to(DEVICE)
        acc, prec, rec, spec, f1 = evaluate(global_model, test_loader)
        print(f"Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | Spec: {spec:.4f} | F1: {f1:.4f}")

        results.append({
            'Model': model_name,
            'Distribution': dist_name,
            'Strategy': strategy,
            'Accuracy': acc,
            'Precision': prec,
            'Recall': rec,
            'Specificity': spec,
            'Macro_F1': f1
        })



## Part 1B: EfficientNet Experiments


In [ ]:
# ==========================================
# EXPERIMENT 2: EFFICIENTNET
# ==========================================
model_name = 'efficientnet'

for dist_name, client_datasets in distributions.items():
    client_loaders = [DataLoader(ds, batch_size=32, shuffle=True) for ds in client_datasets]
    client_sizes = [len(ds) for ds in client_datasets]

    for strategy in strategies:
        print(f"\n--- Running | Model: {model_name.upper()} | Dist: {dist_name.upper()} | Strategy: {strategy.upper()} ---")

        global_model = get_model(model_name)
        global_weights = {k: v.cpu() for k, v in global_model.state_dict().items()}
        fedprox_mu = 0.01 if strategy == 'fedprox' else 0.0

        for rnd in range(NUM_ROUNDS):
            local_weights = []
            for loader in client_loaders:
                local_model = get_model(model_name)
                local_model.load_state_dict(global_weights)
                weights = train_client(local_model, loader, epochs=LOCAL_EPOCHS, lr=2e-4, fedprox_mu=fedprox_mu, global_model=global_model)
                local_weights.append(weights)

            global_weights = aggregate_models(local_weights, strategy, client_sizes)
            global_model.load_state_dict(global_weights)

        global_model.to(DEVICE)
        acc, prec, rec, spec, f1 = evaluate(global_model, test_loader)
        print(f"Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | Spec: {spec:.4f} | F1: {f1:.4f}")

        results.append({
            'Model': model_name,
            'Distribution': dist_name,
            'Strategy': strategy,
            'Accuracy': acc,
            'Precision': prec,
            'Recall': rec,
            'Specificity': spec,
            'Macro_F1': f1
        })



## Part 1C: YOLO Experiments


In [ ]:
# ==========================================
# EXPERIMENT 3: YOLO
# ==========================================
model_name = 'yolo'

for dist_name, client_datasets in distributions.items():
    client_loaders = [DataLoader(ds, batch_size=32, shuffle=True) for ds in client_datasets]
    client_sizes = [len(ds) for ds in client_datasets]

    for strategy in strategies:
        print(f"\n--- Running | Model: {model_name.upper()} | Dist: {dist_name.upper()} | Strategy: {strategy.upper()} ---")

        from ultralytics import YOLO
        yolo_temp = YOLO('yolov8n-cls.pt', verbose=False)
        global_model = yolo_temp.model
        global_model.model[-1].linear = nn.Linear(global_model.model[-1].linear.in_features, NUM_CLASSES)
        global_model = global_model.to(DEVICE)

        global_weights = {k: v.cpu() for k, v in global_model.state_dict().items()}
        fedprox_mu = 0.01 if strategy == 'fedprox' else 0.0

        for rnd in range(NUM_ROUNDS):
            local_weights = []
            for loader in client_loaders:
                yolo_temp = YOLO('yolov8n-cls.pt', verbose=False)
                local_model = yolo_temp.model
                local_model.model[-1].linear = nn.Linear(local_model.model[-1].linear.in_features, NUM_CLASSES)
                local_model = local_model.to(DEVICE)

                local_model.load_state_dict(global_weights)
                weights = train_client(local_model, loader, epochs=LOCAL_EPOCHS, lr=2e-4, fedprox_mu=fedprox_mu, global_model=global_model)
                local_weights.append(weights)

            global_weights = aggregate_models(local_weights, strategy, client_sizes)
            global_model.load_state_dict(global_weights)

        global_model.to(DEVICE)
        acc, prec, rec, spec, f1 = evaluate(global_model, test_loader)
        print(f"Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | Spec: {spec:.4f} | F1: {f1:.4f}")

        results.append({
            'Model': model_name,
            'Distribution': dist_name,
            'Strategy': strategy,
            'Accuracy': acc,
            'Precision': prec,
            'Recall': rec,
            'Specificity': spec,
            'Macro_F1': f1
        })



## Part 2: Client Drift Experiment


In [ ]:
drift_configs = [
    {'Rounds': 10, 'Epochs': 1, 'Label': 'High Comm (R=10, E=1)'},
    {'Rounds': 5,  'Epochs': 2, 'Label': 'Balanced (R=5, E=2)'},
    {'Rounds': 2,  'Epochs': 5, 'Label': 'High Local/Drift (R=2, E=5)'}
]

drift_results = []
drift_model = 'mobilenet'
dist_name = 'non_iid'
strategy = 'fedavg_weighted'

client_datasets = distributions[dist_name]
client_loaders = [DataLoader(ds, batch_size=32, shuffle=True) for ds in client_datasets]
client_sizes = [len(ds) for ds in client_datasets]

print(f"\n{'='*60}\nSTARTING DRIFT EXPERIMENT (MobileNet | Non-IID | FedAvg)\n{'='*60}")

for config in drift_configs:
    print(f"\n--- Testing Config: {config['Label']} ---")
    global_model = get_model(drift_model)
    global_weights = {k: v.cpu() for k, v in global_model.state_dict().items()}

    for rnd in range(config['Rounds']):
        local_weights = []
        for loader in client_loaders:
            local_model = get_model(drift_model)
            local_model.load_state_dict(global_weights)
            weights = train_client(local_model, loader, epochs=config['Epochs'], lr=2e-4, fedprox_mu=0.0, global_model=global_model)
            local_weights.append(weights)

        global_weights = aggregate_models(local_weights, strategy, client_sizes)
        global_model.load_state_dict(global_weights)

    global_model.to(DEVICE)
    acc, prec, rec, spec, f1 = evaluate(global_model, test_loader)
    print(f"Acc: {acc:.4f} | F1: {f1:.4f}")

    drift_results.append({
        'Config': config['Label'],
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'Specificity': spec,
        'Macro_F1': f1
    })



## Part 3: Compile and Plot All Results


In [ ]:
results_df = pd.DataFrame(results)
display(results_df)

metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'Specificity', 'Macro_F1']
fig, axes = plt.subplots(3, 2, figsize=(16, 18))
axes = axes.flatten()

for i, metric in enumerate(metrics_to_plot):
    sns.barplot(data=results_df, x='Model', y=metric, hue='Strategy', errorbar=None, ax=axes[i])
    axes[i].set_title(f'{metric} Comparison Across Models and Strategies')
    axes[i].set_ylabel(metric)
axes[5].set_visible(False)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=results_df, x='Model', y='Macro_F1', hue='Distribution', errorbar=None, ax=axes[0])
axes[0].set_title('Impact of Data Distribution on Macro F1')
axes[0].set_ylabel('Macro F1 Score')

sns.barplot(data=results_df, x='Model', y='Recall', hue='Distribution', errorbar=None, ax=axes[1])
axes[1].set_title('Impact of Data Distribution on Recall')
axes[1].set_ylabel('Recall')
plt.tight_layout()
plt.show()

drift_df = pd.DataFrame(drift_results)
print("\n--- DRIFT EXPERIMENT RESULTS ---")
display(drift_df)

plt.figure(figsize=(10, 6))
sns.barplot(data=drift_df, x='Config', y='Macro_F1', palette='viridis')
plt.title('Client Drift Analysis: Controlled Compute 10-Epoch Trade-off (MobileNet, Non-IID)')
plt.ylabel('Macro F1 Score')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()



## Part 4: Save to Google Drive


In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Convert and Save
# IMPORTANT: Press Ctrl+S (or Cmd+S) to save the notebook's current state before running this cell!
!jupyter nbconvert --to html "/content/drive/MyDrive/Colab Notebooks/Federated_Learning_Split_Execution.ipynb" --output-dir="/content/drive/MyDrive/"

print("\nCheck your Google Drive! The HTML file should be saved in your main MyDrive folder.")

